# Octree Demo

This notebook focuses on octree, lookup, and interpolation behavior.

Scope:

1. Read a data file as a plain `Dataset`
2. Compute per-cell `delta_phi` and inferred refinement levels
3. Build and inspect an octree
4. Run cell lookup in both `xyz` and `(r,polar,azimuth)`
5. Use the interpolator and compare lightly with SciPy


In [1]:
import pooch
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import LinearNDInterpolator
from batread.dataset import Dataset
from batcamp import Octree
from batcamp import OctreeInterpolator
from batcamp.builder import format_histogram
from batcamp.builder_spherical import SphericalOctreeBuilder


### Pick the mixed-level 3D file used during development.


In [2]:
path = pooch.retrieve(
    url="https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=False,
    processor=pooch.Untar(members=["run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt"]),
)
if isinstance(path, (list, tuple)):
    path = path[0]
path


PosixPath('/Users/dagfev/Documents/starwinds/batcamp-main/sample_data/3d__var_2_n00060005.plt')

### Plain Dataset read (no SmartDs)


In [3]:
ds = Dataset.from_file(str(path))
print(ds)


Title:     'BATSRUS: 3D Data, 2000/01/30 00:00:01.000'
Zone:      '3D   N=0060005'
Variables: 4
Shape:     (22016, 4)
Variables: ['X [R]', 'Y [R]', 'Z [R]', 'Rho [g/cm^3]'].


### Build octree and inspect inferred levels


In [4]:
tree = Octree.from_dataset(ds, tree_coord='rpa')
delta_phi, center_phi, cell_levels, expected_delta_phi, coarse_delta_phi = SphericalOctreeBuilder.compute_delta_phi_and_levels(ds)
print('coarsest delta_phi [rad]:', coarse_delta_phi)
print('cell level histogram:', format_histogram(cell_levels))


coarsest delta_phi [rad]: 0.39269923636179715
cell level histogram: 0:1536, 1:2048, 2:16384


### Octree summary


In [5]:
print(tree)


Octree (adaptive): tree_coord=rpa, finest_leaf_grid=(64, 32, 64), root_grid=(2, 1, 2), depth=5, full=True, levels=0..2; leaf_levels[L0:1536 (fine-equiv 98304), L1:2048 (fine-equiv 16384), L2:16384 (fine-equiv 16384)]


### Lookup in xyz and rpa


In [6]:
lookup = tree.lookup
q_xyz = (1.0, 0.0, 0.0)
hit_xyz = lookup.lookup_point(np.array(q_xyz, dtype=float), coord="xyz")
print('lookup xyz:', q_xyz, '->', hit_xyz)
q = np.asarray(q_xyz, dtype=float)
r = float(np.linalg.norm(q))
polar = float(np.arccos(np.clip(q[2] / max(r, float(np.finfo(float).tiny)), -1.0, 1.0)))
azimuth = float(np.mod(np.arctan2(q[1], q[0]), 2.0 * np.pi))
hit_rpa = lookup.lookup_point(np.array([r, polar, azimuth], dtype=float), coord="rpa")
print('lookup rpa:', (r, polar, azimuth), '->', hit_rpa)
print('same cell id?', hit_xyz is not None and hit_rpa is not None and hit_xyz.cell_id == hit_rpa.cell_id)


lookup xyz: (1.0, 0.0, 0.0) -> LookupHit(cell_id=48, level=0, i0=0, i1=4, i2=0, path=((0, 0, 0), (0, 1, 0), (0, 2, 0), (0, 4, 0)), center_xyz=(0.9288220554590225, 0.18475418537855148, -0.19206419587135315))
lookup rpa: (1.0, 1.5707963267948966, 0.0) -> LookupHit(cell_id=48, level=0, i0=0, i1=4, i2=0, path=((0, 0, 0), (0, 1, 0), (0, 2, 0), (0, 4, 0)), center_xyz=(0.9288220554590225, 0.18475418537855148, -0.19206419587135315))
same cell id? True


### Interpolator usage


In [7]:
interp = OctreeInterpolator(
    ds,
    ['Rho [g/cm^3]'],
    tree=tree,
)
queries = np.array([
    [1.0, 0.0, 0.0],
    [5.0, 0.0, 0.0],
    [10.0, 1.0, 0.5],
], dtype=float)
vals, cell_ids = interp(queries, return_cell_ids=True)
print('queries:')
print(queries)
print('values:')
print(vals)
print('cell ids:')
print(cell_ids)


queries:
[[ 1.   0.   0. ]
 [ 5.   0.   0. ]
 [10.   1.   0.5]]
values:
[1.92617789e-13 3.82270565e-20 3.92582511e-21]
cell ids:
[  48  115 5249]


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


### Optional: lightweight comparison to SciPy LinearNDInterpolator.

To keep notebook runtime reasonable, this uses a point subset.


In [8]:
xyz = np.column_stack([
    np.asarray(ds['X [R]'], dtype=float),
    np.asarray(ds['Y [R]'], dtype=float),
    np.asarray(ds['Z [R]'], dtype=float),
])
vv = np.asarray(ds['Rho [g/cm^3]'], dtype=float)
rng = np.random.default_rng(0)
n_sub = min(40000, xyz.shape[0])
sub = rng.choice(xyz.shape[0], size=n_sub, replace=False)
lin = LinearNDInterpolator(xyz[sub], vv[sub], fill_value=np.nan)
q = xyz[rng.choice(xyz.shape[0], size=2000, replace=False)]
ours = interp(q)
scipy_vals = lin(q)
mask = np.isfinite(ours) & np.isfinite(scipy_vals)
print('scipy subset size:', n_sub)
print('finite overlap fraction:', float(np.mean(mask)))
if np.any(mask):
    ad = np.abs(ours[mask] - scipy_vals[mask])
    print('max abs diff:', float(np.max(ad)))
    print('median abs diff:', float(np.median(ad)))


scipy subset size: 22016
finite overlap fraction: 0.9975
max abs diff: 2.297659124121494e-16
median abs diff: 2.0333428914809424e-30
